In [4]:
import os
import json
import webbrowser
import pandas as pd

# (analyze_chunk_lengths 함수 정의부는 기존과 동일합니다)
def analyze_chunk_lengths(folder_path):
    stats_list = []
    if not os.path.exists(folder_path): return None
    files = [f for f in os.listdir(folder_path) if f.endswith('.json') or f.endswith('.jsonl')]
    
    for file_name in files:
        file_path = os.path.join(folder_path, file_name)
        lengths = []
        try:
            if file_name.endswith('.jsonl'):
                with open(file_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        if line.strip():
                            lengths.append(len(str(json.loads(line).get('page_content', ''))))
            else:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                if not isinstance(data, list): data = [data]
                for item in data:
                    lengths.append(len(str(item.get('page_content', ''))))
            if lengths:
                stats_list.append({
                    "File Name": file_name,
                    "Min Len": min(lengths),
                    "Max Len": max(lengths),
                    "Avg Len": round(sum(lengths) / len(lengths), 1)
                })
        except Exception: pass
    return pd.DataFrame(stats_list)

# --- 실행부 ---
target_folder = "../data/04_chunks/final"
df_stats = analyze_chunk_lengths(target_folder)

if df_stats is not None:
    # 💡 데이터프레임을 예쁜 HTML 표로 변환하여 임시 파일로 저장 후 브라우저 오픈
    html_output = "table_result.html"
    
    # 부트스트랩 CSS를 살짝 입혀서 세련된 표 형태로 스타일링
    html_style = """
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css">
    <div class="container mt-4">
        <h2 class="mb-4">📋 청크 길이 통계 결과</h2>
    """
    
    html_table = df_stats.to_html(index=False, classes="table table-striped table-hover table-bordered")
    
    with open(html_output, "w", encoding="utf-8") as f:
        f.write(html_style + html_table + "</div>")
        
    # 기본 웹브라우저로 표 열기
    webbrowser.open('file://' + os.path.realpath(html_output))
    print(f"🎉 브라우저에서 표 형태의 결과를 열었습니다! (파일명: {html_output})")

🎉 브라우저에서 표 형태의 결과를 열었습니다! (파일명: table_result.html)


In [8]:
# 오류가 발생한 파일 경로 지정
error_file = "../data/04_chunks/api/kb_chunks_prec.part01.jsonl"

with open(error_file, 'r', encoding='utf-8') as f:
    first_line = f.readline() # 1번째 줄만 읽기
    
    # 에러가 발생한 319~320자 주변 (앞뒤로 50자씩) 출력해보기
    start = max(0, 319 - 50)
    end = min(len(first_line), 319 + 50)
    
    print("=== 에러 발생 구간 주변 텍스트 ===")
    print(first_line[start:end])
    print(" " * (319 - start) + "^ 여기 부근에서 에러 발생")

=== 에러 발생 구간 주변 텍스트 ===
ce_id": "612861", "chunk_index": 0,"page_content":: "상대방의 의사에 반하여 그의 음성을 녹음하거나, 녹음한 음성을 방송, 배포하는 등의 
                                                  ^ 여기 부근에서 에러 발생


In [13]:
import os
import json
import pandas as pd

def locate_long_chunks_jsonl(folder_path, length_threshold=2000):
    stats_list = []
    
    if not os.path.exists(folder_path):
        print(f"경로를 찾을 수 없습니다: {folder_path}")
        return None
        
    files = [f for f in os.listdir(folder_path) if f.endswith('.jsonl')]
    
    if not files:
        print("폴더 내에 JSONL 파일이 없습니다.")
        return None

    print(f"🔍 각 파일별 {length_threshold}자 이상인 라인 번호 분석 중...")
    print("-" * 80)

    for file_name in files:
        file_path = os.path.join(folder_path, file_name)
        
        long_chunk_lines = []
        total_chunks = 0
        error_count = 0
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                # enumerate를 사용하여 1번째 줄부터 추적 (idx + 1)
                for idx, line in enumerate(f):
                    line = line.strip()
                    if not line:
                        continue
                    
                    total_chunks += 1
                    try:
                        item = json.loads(line)
                        if isinstance(item, dict) and 'page_content' in item:
                            content_len = len(str(item['page_content']))
                            
                            # 2000자 이상인 경우 해당 라인 번호(1부터 시작)를 저장
                            if content_len >= length_threshold:
                                long_chunk_lines.append(idx + 1)
                    except json.JSONDecodeError:
                        error_count += 1
                        continue
            
            # 리스트가 너무 길어지면 표가 깨질 수 있으므로, 
            # 발견된 라인 번호 리스트를 문자열 형태로 저장합니다.
            stats_list.append({
                "file_name": file_name,
                "total_lines": total_chunks,
                "count": len(long_chunk_lines),
                "over_2000_lines": str(long_chunk_lines),  # 라인 번호 목록 위치
                "corrupted": error_count
            })
                
        except Exception as e:
            print(f"파일 접근 오류 ({file_name}): {e}")

    df = pd.DataFrame(stats_list)
    return df

# --- 실행부 ---
target_folder = "../data/04_chunks/api"
df_results = locate_long_chunks_jsonl(target_folder, length_threshold=2000)

if df_results is not None:
    # 표가 화면에 짤리지 않고 끝까지 나오도록 설정
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.width', 1000)
    
    print(df_results.to_string(index=False))

🔍 각 파일별 2000자 이상인 라인 번호 분석 중...
--------------------------------------------------------------------------------
            file_name  total_lines  count over_2000_lines  corrupted
kb_chunks_eflaw.jsonl         2888      0              []          0
